In [1]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as ticker
import seaborn as sns
import os
import re
import json
import unicodedata
from sqlalchemy import create_engine, text

palette_rgb = [(61, 183, 228), (255, 136, 73), (105, 190, 40)]
palette = [(r/255, g/255, b/255) for r, g, b in palette_rgb]

# 1. COMANDOS PARA RECARGA AUTOMÁTICA
%load_ext autoreload
%autoreload 2
%matplotlib inline

pd.set_option('display.float_format', lambda x: f'{x:.2f}') # evitar notación cientifica y mostrar 2 decimales
pd.set_option('display.max_columns', None)   # mostrar todas las columnas
pd.set_option('display.width', 0)           # dejar que use todo el ancho disponible
pd.set_option('display.max_colwidth', None) # Quitar el límite de ancho de las columnas
pd.set_option('display.expand_frame_repr', False) # Para que no "envuelva" la tabla y se mantenga en una sola fila larga


# Descarga de bases de datos:

In [2]:
# Conexión persistente con manejo de timeouts para arquitectura WAL
DB_URL = "sqlite:///../.data/flight_account_001_xauusd.db"
engine = create_engine(DB_URL, connect_args={'timeout': 15})

def extract_raw_tables():
    """
    Extrae la topología completa de la base de datos en DataFrames aislados.
    Retorna un diccionario de DataFrames para acceso modular.
    """
    tables = {
        "unified": "SELECT * FROM unified_department ORDER BY created_at DESC;",
        "layer": "SELECT * FROM analysis_layer;",
        "efficiency": "SELECT * FROM efficiency_audit;",
        "tactical": "SELECT * FROM tactical_audit;"
    }
    
    dataframes = {}
    
    with engine.connect() as conn:
        for name, query in tables.items():
            df = pd.read_sql(text(query), conn)
            
            # Normalización de timestamps si la columna existe en el esquema
            time_cols = [col for col in df.columns if 'time' in col or 'created' in col or 'updated' in col]
            for t_col in time_cols:
                df[t_col] = pd.to_datetime(df[t_col], errors='coerce')
                
            dataframes[name] = df
            
    # Coerción de tipos financieros críticos en Tactical Audit (SQLite NUMERIC -> Pandas Float64)
    if not dataframes["tactical"].empty:
        fin_cols = ['risk_usd', 'size', 'r_r', 'entry_price', 'closing_price', 
                    'take_profit', 'stop_loss', 'pnl_and_cost', 'mae_adverse', 
                    'captured_mae', 'mfe_favorable', 'notional_size', 'capital_at_risk']
        
        # Filtra solo las columnas que realmente existen en el DataFrame extraído
        exist_fin_cols = [c for c in fin_cols if c in dataframes["tactical"].columns]
        dataframes["tactical"][exist_fin_cols] = dataframes["tactical"][exist_fin_cols].apply(pd.to_numeric, errors='coerce')

    return dataframes

# Extracción a memoria RAM
db_data = extract_raw_tables()

# Inspección de cardinalidad y estructura
for table_name, df in db_data.items():
    print(f"\n--- Tabla: {table_name.upper()} ---")
    print(f"Dimensiones (Filas, Columnas): {df.shape}")
    print(f"Columnas detectadas: {list(df.columns)}")


--- Tabla: UNIFIED ---
Dimensiones (Filas, Columnas): (31, 21)
Columnas detectadas: ['id', 'state', 'asset', 'created_at', 'updated_at', 'market_bias', 'calc_edge', 'edge_description', 'trade_status', 'p4_hierarchy', 'p1_timeframe', 'p1_type', 'nodes_l1', 'nodes_l2', 'tactical_classification', 'long_prob', 'short_prob', 'no_trade_prob', 'efficiency_page_id', 'tactical_page_id', 'is_backdated']

--- Tabla: LAYER ---
Dimensiones (Filas, Columnas): (155, 8)
Columnas detectadas: ['id', 'trade_id', 'department', 'layer_name', 'direction', 'strength', 'score', 'thesis']

--- Tabla: EFFICIENCY ---
Dimensiones (Filas, Columnas): (31, 13)
Columnas detectadas: ['id', 'bias_a', 'resolution_type', 'real_bias_b', 'structural_resolution', 'failure_reason', 'specific_bias_compliance', 'false_regime_rate', 'resolution_time', 'lesson_learned', 'created_at', 'updated_at', 'efficiency_timeframe']

--- Tabla: TACTICAL ---
Dimensiones (Filas, Columnas): (30, 42)
Columnas detectadas: ['id', 'compliance', '

/tmp/ipykernel_2380/1208555198.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[t_col] = pd.to_datetime(df[t_col], errors='coerce')
/tmp/ipykernel_2380/1208555198.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[t_col] = pd.to_datetime(df[t_col], errors='coerce')


In [7]:
layer_df = db_data["layer"]
unified_df = db_data["unified"]
efficiency_df = db_data["efficiency"]
tactical_df = db_data["tactical"]

if not tactical_df.empty and not unified_df.empty:
    orphans = tactical_df[~tactical_df['id'].isin(unified_df['id'])]
    if not orphans.empty:
        print(f"ALERTA: Se detectaron {len(orphans)} registros tácticos corruptos (sin Unified ID).")
    else:
        print("Integridad Referencial 1:1 Validada: Todos los registros tácticos corresponden a un Trade matriz.")

Integridad Referencial 1:1 Validada: Todos los registros tácticos corresponden a un Trade matriz.


In [ ]:
### Ver como amarrarlos y las probabilidades.

,id,bias_a,resolution_type,real_bias_b,structural_resolution,failure_reason,specific_bias_compliance,false_regime_rate,resolution_time,lesson_learned,created_at,updated_at,efficiency_timeframe
0,2834cbeb-b65f-4cad-97ca-1807e3193b23,Choppy-Bullish Range Rotation,Confirmed (A equal to B),Choppy-Bullish Range Rotation,Confirmed + expansión significativa,N/A,Valid,True Positive,2026-06-18 09:05:23.445403,"Acá el edge estaba choppy, fue más por intuición el resultado del analisis.",2026-05-18 07:00:00.000000,2026-06-18 09:19:57.914404,NaT
1,1998959d-7ea9-4d3e-8f82-1ec007021143,No_Bias(Choppy),Confirmed (A equal to B),No_Bias(Choppy),N/A,N/A,Valid,True Negative,2026-06-18 09:22:45.685313,None,2026-05-19 06:55:18.949975,2026-06-18 09:23:49.244433,NaT
2,0a68d499-0175-4048-acc4-8412edbb51c4,CHOCH,Confirmed (A equal to B),CHOCH,Confirmed + expansión significativa,N/A,Valid,True Positive,2026-06-21 06:48:35.968186,"El analisis acá estaba cruzado, había posibilidad de que el rango de P0 fuera bajista por la estructura del precio, pero también habia suceptibilidad de que el precio siguiera un pullback para agarrar liquidez y posteriormente seguir bajando.",2026-05-20 06:44:53.674679,2026-06-21 06:53:07.385825,NaT
3,b9c076bf-85fc-450f-a7d2-48917c1652f4,No_Bias(Choppy),Confirmed (A equal to B),No_Bias(Choppy),Confirmed pero mínima,N/A,Valid,True Negative,2026-06-21 07:47:00.664509,"Aqui no hay edge, solo estamos en rango.",2026-05-22 06:55:15.148641,2026-06-21 07:48:11.538823,NaT
4,657c94bb-6383-4e84-b61f-469b6723637f,Choppy-Bullish Range Rotation,Confirmed (A equal to B),Choppy-Bullish Range Rotation,Confirmed pero mínima,N/A,Valid,True Positive,2026-05-24 15:24:31.792157,sfasdfas,2026-05-24 14:50:53.387524,2026-05-24 15:36:24.641780,NaT
5,35be0fc7-b3ba-42d7-9571-63ef2e8100af,Validated Range Expansion,Invalidated (B not equal to A),BOS,N/A,Reversal,Invalid,False Positive,2026-05-26 08:54:16.798760,"El precio estaba en un rango e intenté hacer el trade en el rango superior cuando la tendencia era mayoritariamente bajista, en realidad era más probable el short.",2026-05-25 06:51:45.558733,2026-05-26 19:03:20.065186,NaT
6,22acd14d-4e98-48e5-b6e5-41628f214d3d,Choppy-Bearish Range Rotation,Confirmed (A equal to B),Choppy-Bearish Range Rotation,Confirmed + expansión significativa,N/A,Valid,True Positive,2026-05-26 19:04:33.812837,"Sigue la estructura del precio, el precio rechazó la parte superior del rango con fuerza y se dirigia a la parte inferior con un KL como target, el Bias se cumplió.",2026-05-26 06:35:19.344172,2026-05-26 19:11:47.864681,NaT
7,70ea32f5-6b5b-4cd5-bc9d-a44e58415058,Validated Range Expansion,Confirmed (A equal to B),Validated Range Expansion,Confirmed pero inmediatamente revertido,N/A,Valid,True Positive,2026-05-27 12:33:31.906775,"Aunque este en un rango, al ver como se comportan las subidas y bajadas de entre los niveles superior e inferior puedo ver que tendencia es predominante y si sigue la tendencia principal, hay mayor probabilidad que sea un analisis correcto.",2026-05-27 12:52:10.306744,2026-05-27 12:47:50.015196,NaT
8,5e5b08e7-3107-4c3e-b862-feb929889083,No_Bias(Choppy),Invalidated (B not equal to A),CHOCH,N/A,Range Expansion -- predije choppy y el precio hizo un rompimiento del rango,Invalid,False Negative,2026-06-21 08:01:40.893131,"Los fractales eran bastante claros y mostraban una implicación alcista fuerte, pero la estructura del precio mostraba mucha fuerza bajista. El P3 tenia una descripción correcta y era más suceptible colocar Long-Weak; adicionalmente en P2 debo ver el movimiento en 1H, ya que habia posibilidad de un regreso tras el RSI que mostraba una ""divergencia"" en 1H, pero era debil. En si, el mercado estaba choppy.",2026-05-28 13:04:18.777535,2026-06-21 13:18:28.123794,NaT
9,a4d31d37-7fcb-4296-ab57-53787c960e19,BOS,Confirmed (A equal to B),BOS,Confirmed + expansión significativa,N/A,Valid,True Positive,2026-06-21 13:55:50.225807,"Buen analisis, las probabilidades se cumplieron.",2026-05-29 06:3